In [4]:
import pandas as pd

# ── Load & prep ────────────────────────────────────────────────────────────────
df = pd.read_csv("master.csv")
parts = df["PR Link"].str.split("/", expand=True)
df["app"] = parts[3] + "_" + parts[4]

# ── App exception list ─────────────────────────────────────────────────────────
# Add problematic apps here. These apps will be excluded from sampling.
EXCLUDED_APPS = {
    "code-dot-org_code-dot-org",
    # "another-owner_another-repo",
    # "problematic-app-name_here",
}

df = df[~df["app"].isin(EXCLUDED_APPS)]

df = df[df["Include?"] == 1]

# Shuffle for tie-breaking randomness
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Drop rows missing either required annotation (they can't contribute to coverage)
df = df[df["Broader issue type"].notna() & df["User Demographic"].notna()].reset_index(drop=True)

def parse_field(value):
    """Split a comma-separated field into a cleaned list."""
    return [x.strip() for x in str(value).split(",") if x.strip()]

TARGET = 25
MIN_APPS = 10

# ── Phase 1: coverage-driven selection ────────────────────────────────────────
sampled_rows = []
sample_issue_types = set()
sample_user_demos = set()
sample_apps = set()
used_indices = set()

# Two-pass greedy: first prefer PRs from already-seen apps, then open to new apps.
for prefer_known_app in [True, False]:
    if len(sampled_rows) >= TARGET:
        break

    for i, row in df.iterrows():
        if i in used_indices:
            continue
        if len(sampled_rows) >= TARGET:
            break

        issue_types = parse_field(row["Broader issue type"])
        user_demos  = parse_field(row["User Demographic"])
        app         = row["app"]

        adds_new_issue = any(it not in sample_issue_types for it in issue_types)
        adds_new_demo  = any(d  not in sample_user_demos  for d  in user_demos)

        if not (adds_new_issue or adds_new_demo):
            continue

        # Original app-minimizing rule
        if prefer_known_app and app not in sample_apps and len(sample_apps) > 0:
            continue

        # Reserve enough remaining slots to reach MIN_APPS
        remaining_slots = TARGET - len(sampled_rows)
        missing_apps = max(0, MIN_APPS - len(sample_apps))

        if app in sample_apps and remaining_slots <= missing_apps:
            continue

        sampled_rows.append(row)
        used_indices.add(i)
        sample_issue_types.update(issue_types)
        sample_user_demos.update(user_demos)
        sample_apps.add(app)

# ── Phase 1.5: force minimum app coverage ─────────────────────────────────────
# If coverage selection still used fewer than MIN_APPS apps, add one PR from new apps.
if len(sample_apps) < MIN_APPS:
    for i, row in df.iterrows():
        if i in used_indices:
            continue
        if len(sampled_rows) >= TARGET:
            break
        if len(sample_apps) >= MIN_APPS:
            break

        app = row["app"]

        if app in sample_apps:
            continue

        sampled_rows.append(row)
        used_indices.add(i)
        sample_issue_types.update(parse_field(row["Broader issue type"]))
        sample_user_demos.update(parse_field(row["User Demographic"]))
        sample_apps.add(app)

# ── Phase 2: fill-up to TARGET ─────────────────────────────────────────────────
# Fill remaining slots: first from already-selected apps, then from new apps.
if len(sampled_rows) < TARGET:
    for prefer_known_app in [True, False]:
        if len(sampled_rows) >= TARGET:
            break

        for i, row in df.iterrows():
            if i in used_indices:
                continue
            if len(sampled_rows) >= TARGET:
                break

            app = row["app"]

            if prefer_known_app and app not in sample_apps:
                continue

            sampled_rows.append(row)
            used_indices.add(i)
            sample_issue_types.update(parse_field(row["Broader issue type"]))
            sample_user_demos.update(parse_field(row["User Demographic"]))
            sample_apps.add(app)

# ── Assemble result ────────────────────────────────────────────────────────────
sampled_df = pd.DataFrame(sampled_rows, columns=df.columns).reset_index(drop=True)

# ── Coverage report ────────────────────────────────────────────────────────────
all_issue_types = set(it for v in df["Broader issue type"] for it in parse_field(v))
all_user_demos  = set(d  for v in df["User Demographic"] for d in parse_field(v))

print(f"Excluded apps       : {sorted(EXCLUDED_APPS)}")
print(f"Sample size         : {len(sampled_df)}")
print(f"Distinct apps       : {len(sample_apps)}")
print(f"Issue type coverage : {len(sample_issue_types)}/{len(all_issue_types)}"
      f" ({100*len(sample_issue_types)/len(all_issue_types):.1f}%)")
print(f"User demo coverage  : {len(sample_user_demos)}/{len(all_user_demos)}"
      f" ({100*len(sample_user_demos)/len(all_user_demos):.1f}%)")
print(f"\nApps in sample:\n{sorted(sample_apps)}")
print(f"\nIssue types covered:\n{sorted(sample_issue_types)}")
print(f"\nUser demographics covered:\n{sorted(sample_user_demos)}")

# ── Per-app breakdown ──────────────────────────────────────────────────────────
print("\n── Per-app breakdown ──────────────────────────────────────────────────")
app_issues = {}
app_demos  = {}

for _, row in sampled_df.iterrows():
    app = row["app"]
    app_issues.setdefault(app, set()).update(parse_field(row["Broader issue type"]))
    app_demos.setdefault(app,  set()).update(parse_field(row["User Demographic"]))

for app in sorted(app_issues, key=lambda a: len(app_issues[a]), reverse=True):
    issues = sorted(app_issues[app])
    demos  = sorted(app_demos[app])
    print(f"\n{app}")
    print(f"  Issue types ({len(issues)}): {', '.join(issues)}")
    print(f"  User demographics ({len(demos)}): {', '.join(demos)}")

sampled_df.to_csv("sampled_prs.csv", index=False)
print("\nSaved to sampled_prs.csv")

Excluded apps       : ['code-dot-org_code-dot-org']
Sample size         : 25
Distinct apps       : 10
Issue type coverage : 8/8 (100.0%)
User demo coverage  : 4/4 (100.0%)

Apps in sample:
['Leaflet_Leaflet', 'ONSdigital_design-system', 'adaptlearning_adapt-contrib-boxMenu', 'adaptlearning_adapt_framework', 'cryptpad_cryptpad', 'cylc_cylc-ui', 'nextcloud_photos', 'twbs_bootstrap', 'unl_wdntemplates', 'zotero_zotero']

Issue types covered:
['Color contrast', 'Focus management', 'Heading and content structure', 'Keyboard navigation', 'Motion sensitivity', 'Screen reader information', 'Target size', 'Text presentation']

User demographics covered:
['cognitive impairments', 'general', 'motor impairments', 'visual impairments']

── Per-app breakdown ──────────────────────────────────────────────────

adaptlearning_adapt_framework
  Issue types (4): Focus management, Keyboard navigation, Motion sensitivity, Screen reader information
  User demographics (3): cognitive impairments, motor impai

In [2]:
df=pd.read_csv("sampled_prs.csv")
df[df["PR Id"]==2508890518]

,Labelers,PR Id,PR Link,Issue Summary,Issue type,Detailed Code Change Description,User Demographic,Buggy Lines,Fixed Lines,Notes,Include?,app
15,Kevin & Hasan,2.508891e+09,https://github.com/WebKit/WebKit/pull/33213,The children of iFrame elements were not prope...,"iFrame accessibility, hidden element visibility",Added loop to iterate over the childern of iFr...,general,NaN,NaN,NaN,1,WebKit_WebKit


In [6]:
df[df["PR Id"]==2508890518]["Issue Summary"].tolist()[0]

'The children of iFrame elements were not properly hidden when the isIgnored() attribute is set to the parent iFrame component'

In [ ]:
df[df["PR Id"]==2508890518]["Issue type"].tolist()[0]

In [ ]:
df[df["User Demographic"]==2508890518]["Issue type"].tolist()[0]